# Reproduce TDNet data and train your own models

This notebook is a thin interface to the same package code used by TDNet. It does not ship CFBD data or credentials. You must supply your own CFBD API key and comply with the provider's terms. Generated data, tables, and checkpoints remain in ignored local directories.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'gridiron_ml').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Run this notebook from inside a TDNet checkout.')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
print(PROJECT_ROOT)

## 1. Fetch source data

Set `CFBD_API_KEY` in the environment that launches Jupyter. Never paste a real key into this notebook. Adjust the year range as needed. The fetcher is cache-aware.

In [ ]:
START_YEAR = 2010
END_YEAR = 2025
if not os.environ.get('CFBD_API_KEY'):
    raise RuntimeError('CFBD_API_KEY is not set in the Jupyter environment.')

env = os.environ.copy()
env.update({
    'START_YEAR': str(START_YEAR),
    'END_YEAR': str(END_YEAR),
    'OUTPUT_ROOT': str(PROJECT_ROOT / 'data' / 'raw' / 'cfbd' / 'v2'),
})
subprocess.run([str(PROJECT_ROOT / 'scripts' / 'fetch_cfbd_release_data.sh')], cwd=PROJECT_ROOT, env=env, check=True)

## 2. Review the training configuration

The checked-in configuration defines the train, validation, and test seasons and the enabled model families. Copy it before changing the canonical project configuration when running a custom experiment.

In [ ]:
from gridiron_ml import TDRun

CONFIG = PROJECT_ROOT / 'configs' / 'td_run' / 'data_and_train.yaml'
runner = TDRun.from_config(CONFIG)
runner.config

## 3. Build fingerprints and train

These calls can be compute-intensive. The pipeline writes derived data locally; training writes model artifacts under the configured model root. Market columns are rejected unless a configuration explicitly opts into a market-aware experiment.

In [ ]:
pipeline_result = runner.run_data_pipeline()
training_result = runner.train_models()
training_result

## 4. Evaluate locally

Evaluation tables stay under ignored local output paths. Select figures may be copied into the publication tree only after reviewing them for data redistribution and provenance.

In [ ]:
evaluation_result = runner.evaluate_latest_checkpoints()
evaluation_result